In [1]:
import os
import glob
import json
import csv
import math
import numpy as np
from tqdm import tqdm

In [11]:
root_path = 'results'
save_path = os.path.join(root_path, "sorted")
os.makedirs(save_path, exist_ok = True)

task_type_list = ['FiniteList', 'FiniteSet', 'FiniteStack', 'MaxNum', 'Nback', 'WCST']
#task_type_list = ['MaxNum']

In [14]:
for task_type in tqdm(task_type_list, desc = "ongoing", leave = False):

    path_list = glob.glob(os.path.join(root_path, "*", task_type, "*"))
    
    #decodability(path_list)
    #cond_decodability(path_list)
    #separability(path_list)
    #rev_cond_decodability(path_list)

    #clustering(path_list)
    #alignment(path_list)
    #axis_removal(path_list)
    state_confusion(path_list)

ongoing:   0%|                                                                                   | 0/6 [00:00<?, ?it/s]

k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays


ongoing:  33%|█████████████████████████                                                  | 2/6 [00:00<00:01,  3.33it/s]

k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays
k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays


ongoing:  50%|█████████████████████████████████████▌                                     | 3/6 [00:01<00:01,  2.40it/s]

k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays


ongoing:  67%|██████████████████████████████████████████████████                         | 4/6 [00:03<00:02,  1.03s/it]

k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays


ongoing: 100%|███████████████████████████████████████████████████████████████████████████| 6/6 [00:04<00:00,  1.14it/s]

k acc_confusion_ifchanges
k acc_confusion_ifstays
k acc_original_ifchanges
k acc_original_ifstays


In [2]:
def decodability(path_list):

    final_dict = dict()

    first_flag = True
    for path in path_list:

        with open(os.path.join(path, f'{task_type}_decodability.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)
        
        if first_flag:
            for k, v in data.items():
                final_dict[k] = v
            first_flag = False
        else:
            for k, v in data.items():
                final_dict[k].extend(v)

    curr_data = dict(sorted(final_dict.items(), key = lambda x: sum(x[1]) / len(x[1])))
    
    max_len = max(len(v) for v in curr_data.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_decodability.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(curr_data.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [curr_data[k][i] if i < len(curr_data[k]) else '' for k in headers]
            writer.writerow(row)

In [3]:
# cond_decodability

def cond_decodability(path_list):

    final_dict = dict()

    first_flag = True
    for path in path_list:

        with open(os.path.join(path, f'{task_type}_cond_decodability.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)
        
        if first_flag:
            for k, v in data.items():
                final_dict[k] = v["cog_state"]
            first_flag = False
        else:
            for k, v in data.items():
                final_dict[k].extend(v["cog_state"])

    curr_data = dict(sorted(final_dict.items(), key = lambda x: sum(x[1]) / len(x[1])))
    
    max_len = max(len(v) for v in curr_data.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_cond_decodability.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(curr_data.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [curr_data[k][i] if i < len(curr_data[k]) else '' for k in headers]
            writer.writerow(row)

In [4]:
# separability

def separability(path_list):

    final_dict = dict()
    threshold = 5

    first_flag = True
    for path in path_list:

        with open(os.path.join(path, f'{task_type}_separability.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)

        filtered_separability = {}
        excluded_counts = {}
        
        for key, values in data.items():
        
            valid_values = [value for value in values if np.isfinite(value) and value < threshold]
            excluded_counts[key] = [value for value in values if value not in valid_values]
        
            if len(valid_values) > 0:
                filtered_separability[key] = valid_values
        
        if first_flag:
            for k, v in filtered_separability.items():
                final_dict[k] = v
            first_flag = False
        else:
            for k, v in filtered_separability.items():
                final_dict[k].extend(v)

    curr_data = dict(sorted(final_dict.items(), key = lambda x: sum(x[1]) / len(x[1])))
    
    max_len = max(len(v) for v in curr_data.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_separability.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(curr_data.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [curr_data[k][i] if i < len(curr_data[k]) else '' for k in headers]
            writer.writerow(row)

In [5]:
# rev_cond_decodability

def rev_cond_separability(path_list):
    
    final_dict = dict()

    first_flag = True
    for path in path_list:

        with open(os.path.join(path, f'{task_type}_rev_cond_decodability.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)

        inner = list(data.items())
        inner = inner[0][1]
    
        for k, v in list(inner.items()):
            if len(v) == 0:
                inner[k] = [float(0)]
            else:
                if any(isinstance(x, float) and math.isnan(x) for x in v):
                    print(task_type)
                inner[k] = [0 if (isinstance(x, float) and math.isnan(x)) else x for x in v]
        
        if first_flag:
            for k, v in inner.items():
                final_dict[k] = v
            first_flag = False
        else:
            for k, v in inner.items():
                final_dict[k].extend(v)

    curr_data = dict(sorted(final_dict.items(), key = lambda x: sum(x[1]) / len(x[1])))
    
    max_len = max(len(v) for v in curr_data.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_rev_cond_decodability.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(curr_data.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [curr_data[k][i] if i < len(curr_data[k]) else '' for k in headers]
            writer.writerow(row)

In [6]:
# clustering

def clustering(path_list):
    
    final_dict = dict()

    first_flag = True
    for path in path_list:
    
        with open(os.path.join(path, f'{task_type}_clustering.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)
        
        if first_flag:
            for k, v in data.items():
                final_dict[k] = {k_inner: [v[k_inner]] for k_inner in v.keys() if "_k_" not in str(k_inner) and "_eps_" not in str(k_inner)}
            first_flag = False
        else:
            for k, v in data.items():
                final_dict[k] = {k_inner: final_dict[k][k_inner] + [v[k_inner]] for k_inner in final_dict[k].keys() if k_inner in final_dict[k].keys()}

    rows = []
    for feature, methods in final_dict.items():
        for method, values in methods.items():
            for value in values:
                rows.append([feature, method, value])
    
    df = pd.DataFrame(rows, columns = ["Feature", "Method", "Value"])
    df.to_csv(os.path.join(save_path, f'{task_type}_sorted_clustering.csv'), index = False)

In [7]:
# alignment

def alignment(path_list):

    final_dict = {key: [] for key in ["epoch", "train_acc", "val_acc", 
                                      "epoch_acc_mean", "epoch_acc_std", "epoch_bal_acc_mean", "epoch_bal_acc_std"]}

    for path in path_list:
    
        with open(os.path.join(path, f'{task_type}_alignment.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)

        # loss change
        for epoch in data["log"].keys():

            final_dict["epoch"].append(epoch)
            final_dict["train_acc"].append(data["log"][str(epoch)]["train_acc"])
            final_dict["val_acc"].append(data["log"][str(epoch)]["val_acc"])
            
            final_dict["epoch_acc_mean"].append(np.mean(data["epoch_acc"]['SVM'][str(epoch)]["scores_acc"]))
            final_dict["epoch_acc_std"].append(np.std(data["epoch_acc"]['SVM'][str(epoch)]["scores_acc"]))

            final_dict["epoch_bal_acc_mean"].append(np.mean(data["epoch_acc"]['SVM'][str(epoch)]["scores_bal_acc"]))
            final_dict["epoch_bal_acc_std"].append(np.std(data["epoch_acc"]['SVM'][str(epoch)]["scores_bal_acc"]))

    max_len = max(len(v) for v in final_dict.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_alignment.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(final_dict.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [final_dict[k][i] if i < len(final_dict[k]) else '' for k in headers]
            writer.writerow(row)

In [9]:
# axis removal

def axis_removal(path_list):

    final_dict = dict()

    first_flag = True
    for path in path_list:
    
        with open(os.path.join(path, f'{task_type}_axis_removal.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)
        
        if first_flag:
            for k, v in data.items():
                final_dict[k] = [v] if isinstance(v, float) else v
            first_flag = False
        else:
            for k, v in data.items():
                final_dict[k].extend([v] if isinstance(v, float) else v)
    
    max_len = max(len(v) for v in final_dict.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_axis_removal.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(final_dict.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [final_dict[k][i] if i < len(final_dict[k]) else '' for k in headers]
            writer.writerow(row)

In [13]:
# state confusion

def state_confusion(path_list):

    final_dict = dict()

    first_flag = True
    for path in path_list:
    
        with open(os.path.join(path, f'{task_type}_state_confusion.json'), "r", encoding = "utf-8") as file:
            data = json.load(file)
        
        if first_flag:
            for k, v in data.items():
                print("k", k)
                final_dict[k + "_mean"] = [np.mean(v)]
                final_dict[k + "_std"] = [np.std(v)]
            first_flag = False
        else:
            for k, v in data.items():
                final_dict[k + "_mean"].append(np.mean(v))
                final_dict[k + "_std"].append(np.std(v))
    
    max_len = max(len(v) for v in final_dict.values())
    with open(os.path.join(save_path, f'{task_type}_sorted_state_confusion.csv'), 'w', newline = '', encoding = 'utf-8') as f:
        
        writer = csv.writer(f)
        
        headers = list(final_dict.keys())
        writer.writerow(headers)
        
        for i in range(max_len):
            row = [final_dict[k][i] if i < len(final_dict[k]) else '' for k in headers]
            writer.writerow(row)